# Primer EDA — Despacho de energía con RL

Proyecto Final SI3003 — Universidad EAFIT — Idea 3.3

**Objetivo del hito (Mié 4 – Jue 5 nov):** verificar acceso a datos reales de XM (precio de bolsa) y hacer un primer análisis exploratorio. Si la API de XM falla, se usa el plan B (CSV local en `data/raw/`).

Integrante: Joan Stiven Peralta Bedoya

In [ ]:
import sys
from pathlib import Path
from datetime import date

import pandas as pd
import matplotlib.pyplot as plt

# Permite importar el paquete src cuando el notebook corre desde notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
print('Raíz del proyecto:', ROOT)

## 1. Cargar el precio de bolsa (API de XM con plan B)

Rango de ejemplo: primer trimestre de 2024. Ajusta las fechas según disponibilidad.

In [ ]:
try:
    from src.data.download_xm import obtener_precio_bolsa
    df = obtener_precio_bolsa(date(2024, 1, 1), date(2024, 3, 31))
except Exception as e:
    print('No se pudo cargar vía módulo/API:', e)
    print('Coloca un CSV en data/raw/precio_bolsa.csv (plan B) y reejecuta.')
    df = pd.DataFrame()

df.head()

## 2. Estructura y calidad de los datos

In [ ]:
if not df.empty:
    print('Dimensiones:', df.shape)
    print('\nColumnas:', list(df.columns))
    print('\nTipos:')
    print(df.dtypes)
    print('\nNulos por columna:')
    print(df.isna().sum())
    display(df.describe(include='all'))

## 3. Visualización 1 — Serie de tiempo del precio de bolsa

Nota: la API de XM devuelve el precio por horas (columnas `Values_Hour01`..`Values_Hour24`) y una fila por día. Según el formato real, puede ser necesario *derretir* (melt) a formato largo. Esta celda es un esqueleto a ajustar tras ver las columnas reales.

In [ ]:
if not df.empty:
    hour_cols = [c for c in df.columns if 'Hour' in str(c)]
    if hour_cols and 'Date' in df.columns:
        largo = df.melt(id_vars=['Date'], value_vars=hour_cols,
                        var_name='hora', value_name='precio')
        largo['hora'] = largo['hora'].str.extract(r'(\d+)').astype(int)
        largo['ts'] = pd.to_datetime(largo['Date']) + pd.to_timedelta(largo['hora'] - 1, unit='h')
        largo = largo.sort_values('ts')
        fig, ax = plt.subplots(figsize=(14, 4))
        ax.plot(largo['ts'], largo['precio'], lw=0.6)
        ax.set_title('Precio de bolsa (horario)')
        ax.set_xlabel('Fecha'); ax.set_ylabel('Precio [$/kWh]')
        plt.tight_layout(); plt.show()
    else:
        print('Ajusta los nombres de columna según el formato real de pydataxm.')

## 4. Visualización 2 — Perfil horario promedio

Clave para el MDP: muestra las horas baratas (cargar) y caras (descargar).

In [ ]:
if not df.empty and 'largo' in dir():
    perfil = largo.groupby('hora')['precio'].mean()
    fig, ax = plt.subplots(figsize=(10, 4))
    perfil.plot(kind='bar', ax=ax)
    ax.set_title('Precio promedio por hora del día')
    ax.set_xlabel('Hora'); ax.set_ylabel('Precio promedio [$/kWh]')
    plt.tight_layout(); plt.show()

## 5. Conclusiones del primer EDA

_(A completar tras ejecutar con datos reales)_

- ¿Se verificó el acceso a la API de XM? ¿Hizo falta el plan B?
- Rango de fechas disponible y granularidad (horaria).
- Volatilidad del precio y patrón horario → informa el diseño del MDP (bandas de precio).
- Datos faltantes / outliers a limpiar antes de entrenar la Parte 1.